<a href="https://colab.research.google.com/github/vunhankhanhmcs3-cmd/master-thesis-mooc-recommendation/blob/main/05_RL_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import math
import pickle
from collections import defaultdict
from tqdm import tqdm
from google.colab import drive

In [ ]:
# 1. Cấu hình
drive.mount('/content/drive')

# Đường dẫn (Cần đảm bảo bạn đã upload file raw vào đây)
BASE_PATH = '/content/drive/MyDrive/01. THESIS/My suggestion/'
PROCESSED_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/processed/')
RELATIONS_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/raw/relations/')
if not os.path.exists(PROCESSED_PATH): os.makedirs(PROCESSED_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- CẤU HÌNH THỰC NGHIỆM ĐÁNH GIÁ LUẬN VĂN ---
TOP_K = 10           # Top-K khóa học gợi ý tối ưu
NUM_RUNS = 5         # Số lần lặp lại thí nghiệm để tính toán độ lệch chuẩn (Std Dev) chuẩn học thuật
MAX_STEPS = 4        # Số bước đi tối đa của Agent (Đồng bộ khớp với File 04)
NUM_TEST_USERS = 0   # 0 = Cấu hình chạy trên TOÀN BỘ người dùng kiểm thử để lấy số liệu thực tế

print(f"⚙️ Cấu hình thực nghiệm Pha Đánh giá: Runs={NUM_RUNS}, Path Length={MAX_STEPS}, Top-K={TOP_K}")

⚙️ Cấu hình thực nghiệm Pha Đánh giá: Runs=5, Path Length=4, Top-K=10


In [ ]:
# 1. Khởi tạo lại các Lớp đối tượng (Bắt buộc để nạp lại Checkpoint weights)
class MOOCEnvironment:
    def __init__(self, data_path):
        self.ent_emb = np.load(os.path.join(data_path, 'entity_embeddings.npy'))
        self.rel_emb = np.load(os.path.join(data_path, 'relation_embeddings.npy'))
        kg = pd.read_csv(os.path.join(data_path, 'kg_final.txt'), sep=' ', header=None)
        self.graph = defaultdict(list)
        for _, row in kg.iterrows():
            self.graph[int(row[0])].append((int(row[1]), int(row[2])))

        with open(os.path.join(data_path, 'entity_stats.txt'), 'r') as f:
            stats = {l.split('=')[0]: int(l.split('=')[1]) for l in f}
        self.course_start = stats['user_num']
        self.course_end = stats['user_num'] + stats['course_num']
        self.visited = set()

    def reset(self, uid):
        self.state = uid
        self.path = [(uid, -1)]
        self.visited = {uid}
        return self.get_state()

    def get_state(self):
        uid = self.path[0][0]
        return np.concatenate([self.ent_emb[uid], self.ent_emb[self.state]])

    def get_valid_actions(self):
        acts = self.graph[self.state]
        return [(r, n) for r, n in acts if n not in self.visited]

    def step(self, action):
        r, n = action
        self.state = n
        self.path.append((n, r))
        self.visited.add(n)
        return self.get_state(), 0, False, self.path

class PolicyNet(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.l1 = nn.Linear(s_dim + a_dim, 128)
        self.l2 = nn.Linear(128, 1)

    def forward(self, state, action_emb):
        x = torch.cat([state.unsqueeze(1).expand(-1, action_emb.size(1), -1), action_emb], -1)
        x = F.elu(self.l1(x))
        return self.l2(x).squeeze(-1)

In [ ]:
# 2. Tải mô hình và chuẩn bị tệp dữ liệu phân tách từ bước 8:1:1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🧠 Đang sử dụng phần cứng tăng tốc: {device}")

env = MOOCEnvironment(PROCESSED_PATH)
net = PolicyNet(200, 100).to(device)

model_path = os.path.join(PROCESSED_PATH, 'policy_net.pth')
try:
    net.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
except:
    net.load_state_dict(torch.load(model_path, map_location=device))
net.eval()

# A. Nạp lịch sử học tập (Lịch sử hóa tập Train để loại bỏ các khóa học cũ người học đã biết)
train_df = pd.read_csv(os.path.join(PROCESSED_PATH, 'train.txt'), sep=' ', header=None, names=['u','c','l'])
train_df['c'] += env.course_start
user_hist = train_df.groupby('u')['c'].apply(set).to_dict()

# B. Nạp tập Kiểm thử mục tiêu (Test Ground Truth từ sản phẩm file 02)
test_df = pd.read_csv(os.path.join(PROCESSED_PATH, 'test.txt'), sep=' ', header=None, names=['u','c','l'])
test_df['c'] += env.course_start
test_target = test_df.groupby('u')['c'].apply(set).to_dict()

# C. Cấu hình tập mẫu người dùng kiểm thử
target_users = list(test_target.keys())
if NUM_TEST_USERS > 0:
    target_users = target_users[:NUM_TEST_USERS]

print(f"🚀 Hệ thống sẵn sàng chạy Pha suy luận trên TOÀN BỘ {len(target_users)} người dùng kiểm thử...")

🧠 Đang sử dụng phần cứng tăng tốc: cpu
🚀 Hệ thống sẵn sàng chạy Pha suy luận trên TOÀN BỘ 6486 người dùng kiểm thử...


In [ ]:
# 3. Định nghĩa Quy trình đánh giá trên từng lượt chạy mẫu (Single Run Loop)
def evaluate_one_run(run_id):
    metrics = {'ndcg': [], 'hr': [], 'precision': [], 'recall': []}
    invalid_users = 0
    case_studies = []

    pbar = tqdm(target_users, desc=f"Lượt chạy (Run) {run_id+1}/{NUM_RUNS}")
    for uid in pbar:
        recs = defaultdict(int)
        hist = user_hist.get(uid, set())
        paths = {}

        # Thực hiện kỹ thuật lấy mẫu 50 lộ trình trên mỗi người học để đa dạng hóa (Path Sampling)
        for _ in range(50):
            state = env.reset(uid)
            for _ in range(MAX_STEPS):
                valid = env.get_valid_actions()
                if not valid: break

                act_vecs = []
                for r, n in valid: act_vecs.append(env.rel_emb[r] + env.ent_emb[n])
                act_t = torch.tensor(np.array(act_vecs)).float().to(device).unsqueeze(0)
                state_t = torch.tensor(state).float().to(device).unsqueeze(0)

                # Khai phá không gian hành động bằng mạng chính sách
                with torch.no_grad():
                    probs = F.softmax(net(state_t, act_t), 1)
                    idx = torch.multinomial(probs, 1).item()

                state, _, _, path = env.step(valid[idx])
                node = path[-1][0]

                # Nếu tác nhân chạm đến thực thể Khóa học mới
                if env.course_start <= node < env.course_end:
                    if node not in hist:
                        recs[node] += 1
                        paths[node] = path
                        break

        # Trích xuất danh sách Top-K dựa trên tần suất phân phối (Frequency Ranking)
        top_k = sorted(recs, key=recs.get, reverse=True)[:TOP_K]

        # Xử lý các tình huống đặc biệt khi Agent không tìm thấy lộ trình hợp lệ
        if not top_k:
            invalid_users += 1
            metrics['ndcg'].append(0)
            metrics['hr'].append(0)
            metrics['precision'].append(0)
            metrics['recall'].append(0)
            continue

        truth = test_target[uid]
        hits = len(set(top_k) & truth)

        # Tính toán chỉ số định lượng trên từng cá nhân người học
        metrics['hr'].append(1 if hits > 0 else 0)
        metrics['precision'].append(hits / TOP_K)
        metrics['recall'].append(hits / len(truth))

        dcg = sum([1/math.log2(i+2) for i, x in enumerate(top_k) if x in truth])
        idcg = sum([1/math.log2(i+2) for i in range(min(len(truth), TOP_K))])
        metrics['ndcg'].append(dcg/idcg if idcg > 0 else 0)

        # Lưu trữ nghiên cứu tình huống giải thích (Case Study) tại lượt chạy đầu tiên
        if run_id == 0:
            for item in top_k:
                if item in truth:
                    case_studies.append({'uid': uid, 'target_course': item, 'path': paths[item]})
                    break

    # Tính giá trị trung bình nhân với hệ số 100 để đưa về đơn vị %
    avg_metrics = {k: np.mean(v) * 100 for k, v in metrics.items()}
    invalid_rate = (invalid_users / len(target_users)) * 100

    return avg_metrics, invalid_rate, case_studies

In [ ]:
# 4. Vòng lặp Tổng hợp Đa tầng (Multiple Runs Validation Pipeline)
final_results = {'ndcg': [], 'hr': [], 'precision': [], 'recall': [], 'invalid': []}
saved_case_studies = []

print(f"🔥 Bắt đầu quy trình kiểm thử hệ thống lặp lại {NUM_RUNS} lần...")
for i in range(NUM_RUNS):
    res, inv, cases = evaluate_one_run(i)

    final_results['ndcg'].append(res['ndcg'])
    final_results['hr'].append(res['hr'])
    final_results['precision'].append(res['precision'])
    final_results['recall'].append(res['recall'])
    final_results['invalid'].append(inv)

    if i == 0:
        saved_case_studies = cases

🔥 Bắt đầu quy trình kiểm thử hệ thống lặp lại 5 lần...


Lượt chạy (Run) 5/5: 100%|██████████| 6486/6486 [29:23<00:00,  3.68it/s]


In [ ]:
print("\n" + "="*70)
print(f"📊 BÁO CÁO KẾT QUẢ THỰC NGHIỆM ĐỊNH LƯỢNG (Path Length={MAX_STEPS})")
print("="*70)
print(f"{'Chỉ số (Metric)':<18} | {'Trung bình (Mean)':<12} | {'Độ lệch chuẩn (Std)':<12} | {'Định dạng (Mean ± Std)'}")
print("-" * 70)

summary_text = "==================================================\n"
summary_text += f"BÁO CÁO ĐÁNH GIÁ MÔ HÌNH TRÊN TOÀN BỘ TEST USER\n"
summary_text += "==================================================\n"
summary_text += f"Quy mô kiểm thử: {len(target_users)} người học\n"
summary_text += f"Độ dài bước nhảy (Max Steps): {MAX_STEPS}\n"
summary_text += "-" * 50 + "\n"

metric_names = ['ndcg', 'recall', 'hr', 'precision', 'invalid']
display_names = ['NDCG@10', 'Recall@10', 'Hit Rate@10', 'Precision@10', 'Tỷ lệ Invalid']

for key, name in zip(metric_names, display_names):
    values = final_results[key]
    mean_val = np.mean(values)
    std_val = np.std(values)

    if key == 'invalid':
        display_str = f"{mean_val:.2f}% ± {std_val:.2f}"
    else:
        display_str = f"{mean_val:.2f} ± {std_val:.2f}"

    print(f"{name:<18} | {mean_val:<12.4f} | {std_val:<12.4f} | {display_str}")
    summary_text += f"{name:<18}: {display_str}\n"

print("="*70)


📊 BÁO CÁO KẾT QUẢ THỰC NGHIỆM ĐỊNH LƯỢNG (Path Length=4)
Chỉ số (Metric)    | Trung bình (Mean) | Độ lệch chuẩn (Std) | Định dạng (Mean ± Std)
----------------------------------------------------------------------
NDCG@10            | 3.5948       | 0.0665       | 3.59 ± 0.07
Recall@10          | 5.9627       | 0.0855       | 5.96 ± 0.09
Hit Rate@10        | 11.0268      | 0.2146       | 11.03 ± 0.21
Precision@10       | 1.1665       | 0.0232       | 1.17 ± 0.02
Tỷ lệ Invalid      | 0.0000       | 0.0000       | 0.00% ± 0.00


In [ ]:
# 6. Lưu trữ sản phẩm thực nghiệm phục vụ viết Chương 4
with open(os.path.join(PROCESSED_PATH, 'case_studies.pkl'), 'wb') as f:
    pickle.dump(saved_case_studies, f)

with open(os.path.join(PROCESSED_PATH, 'final_results2.txt'), 'w', encoding='utf-8') as f:
    f.write(summary_text)

print("✅ Hoàn tất lưu trữ 'case_studies.pkl' và báo cáo văn bản 'final_results2.txt'.")
print("👉 Bạn có thể mở file 'final_results2.txt' trên Drive để chép trực tiếp số liệu vào Bảng 4.2 của luận văn.")

✅ Hoàn tất lưu trữ 'case_studies.pkl' và báo cáo văn bản 'final_results2.txt'.
👉 Bạn có thể mở file 'final_results2.txt' trên Drive để chép trực tiếp số liệu vào Bảng 4.2 của luận văn.
